# employee_bronze_to_silver
Reads CSVs from source_to_bronze with custom schema, transforms, writes Delta to silver

In [0]:
%run /Workspace/Users/swati.k@diggibyte.com/databricks-assignment/src/Question_1/source_to_bronze/utils

In [0]:

employee_bronze_path   = '/Volumes/workspace/default/upload/source_to_bronze/employee'
department_bronze_path = '/Volumes/workspace/default/upload/source_to_bronze/department'
country_bronze_path    = '/Volumes/workspace/default/upload/source_to_bronze/country'

print('Bronze paths set')

In [0]:

from pyspark.sql.types import *

employee_schema = StructType([
    StructField('EmployeeID',   IntegerType(), True),
    StructField('EmployeeName', StringType(),  True),
    StructField('Department',   StringType(),  True),
    StructField('Country',      StringType(),  True),
    StructField('Salary',       DoubleType(),  True),
    StructField('Age',          IntegerType(), True)
])

print('Employee schema defined')
print('   EmployeeID   -> IntegerType')
print('   EmployeeName -> StringType')
print('   Department   -> StringType  (e.g. D101)')
print('   Country      -> StringType  (e.g. IN)')
print('   Salary       -> DoubleType')
print('   Age          -> IntegerType')

In [0]:

employee_df = spark.read.format('csv') \
    .option('header', 'true') \
    .schema(employee_schema) \
    .load(employee_bronze_path)

print(' employee_df loaded with custom schema')
print('Row count:', employee_df.count())
employee_df.printSchema()
display(employee_df)

In [0]:

department_schema = StructType([
    StructField('DepartmentID',   StringType(), True),
    StructField('DepartmentName', StringType(), True)
])

department_df = spark.read.format('csv') \
    .option('header', 'true') \
    .schema(department_schema) \
    .load(department_bronze_path)

print('department_df loaded')
print('Row count:', department_df.count())
display(department_df)

In [0]:

country_schema = StructType([
    StructField('CountryCode', StringType(), True),
    StructField('CountryName', StringType(), True)
])

country_df = spark.read.format('csv') \
    .option('header', 'true') \
    .schema(country_schema) \
    .load(country_bronze_path)

print('country_df loaded')
print('Row count:', country_df.count())
display(country_df)

In [0]:

employee_df = convert_to_snake_case(employee_df)

print('snake_case columns applied:')
print(employee_df.columns)

In [0]:

employee_df = add_load_date(employee_df)

print(' load_date column added')
display(employee_df)

In [0]:

spark.sql('CREATE DATABASE IF NOT EXISTS Employee_info')
print(' Database Employee_info ready')

In [0]:

silver_path = '/Volumes/workspace/default/upload/silver/Employee_info/dim_employee'

print('Silver path:', silver_path)

In [0]:

employee_df.write \
    .format('delta') \
    .mode('overwrite') \
    .save(silver_path)

print(' Delta table written to silver layer at:', silver_path)

In [0]:

spark.read.format('delta') \
    .load(silver_path) \
    .createOrReplaceTempView('dim_employee')

print('Temp view dim_employee created')
print('Final silver table:')
display(spark.sql('SELECT * FROM dim_employee'))